### 【字體載入】
- 只執行一次就永久有效

In [1]:
%%html
<link href="https://fonts.googleapis.com/css2?family=Shippori+Mincho:wght@500;700&family=Zen+Antique&family=Noto+Serif+TC:wght@400;700&family=Kingnamai-Handwriting&display=swap" rel="stylesheet">

In [1]:
# Cell 1：載入所有套件
import torch
import torch.nn as nn
import numpy as np
import cv2
import mediapipe as mp
from pathlib import Path
import json
import time
import os
from IPython.display import display, HTML, clear_output
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from accelerate import init_empty_weights
from datetime import datetime
from collections import deque

print("所有套件載入完成！準備喚醒AI……")

所有套件載入完成！準備喚醒AI……


In [2]:
# Cell 2：載入你的動作編碼器
class MotionEncoder(nn.Module):
    def __init__(self, input_dim=177, hidden_dim=256, embed_dim=256, num_layers=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=0.3, bidirectional=True)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim*2, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, embed_dim)
        )
    
    def forward(self, x):
        out, (h, c) = self.lstm(x)
        emb = torch.cat([h[-2], h[-1]], dim=-1)
        return self.proj(emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = MotionEncoder().to(device)
encoder.load_state_dict(torch.load("weights/lstm_encoder_best.pth", map_location=device))
encoder.eval()
print("動作編碼器載入成功！")

動作編碼器載入成功！


C:\Users\AW'z\AppData\Local\Temp\ipykernel_14736\3564320624.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encoder.load_state_dict(torch.load("weights/lstm_encoder_bes

### 【Model Name】
#### CPU
- microsoft/Phi-3-mini-4k-instruct
#### GPU
- MediaTek-Research/Breeze-7B-Instruct-v0_1
- MediaTek-Research/Breeze-7B-Instruct-v1_0

In [3]:
#Cell 3
model_name = "MediaTek-Research/Breeze-7B-Instruct-v1_0"

print("正在載入 Breeze-7B 4bit（穩定版）...")

# 量化設定（用 float16 避免 bfloat16 問題）
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 單一載入呼叫：**不使用 init_empty_weights**（這是錯誤根源！）
# 直接用 device_map + quantization_config，讓 accelerate 自動處理
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",                    # KeyPoint：讓 accelerate 處理分層
    dtype=torch.float16,                  # 用 dtype 而非 torch_dtype（解決警告）
    trust_remote_code=True,
    cache_dir="./breeze_cache",
    # 不要加 low_cpu_mem_usage=True（會衝突）
    # 不要加 offload_folder（除非你真的要 offload）
)

print("Breeze-7B 4bit 載入成功！")
print(f"GPU 記憶體：{torch.cuda.memory_allocated() / 1024**3:.2f} GB")

正在載入 Breeze-7B 4bit（穩定版）...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Breeze-7B 4bit 載入成功！
GPU 記憶體：4.32 GB


In [4]:
# Cell 3.5：測試生成
messages = [{"role": "user", "content": "Hi, Breeze!!"}]
input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = model.generate(input_ids, max_new_tokens=50, temperature=0.7, do_sample=True)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print("Test Response：", response)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Test Response：  You are a helpful AI assistant built by MediaTek Research. The user you are helping speaks Traditional Chinese and comes from Taiwan.  [INST] Hi, Breeze!! [/INST] 嗨，你好！很高興可以幫你提供一些資訊。有任何需求或問題，請不要猶豫，直接問我。


In [5]:
# Cell 4：動作特徵提取 —— 完全對齊你訓練時的 177 維
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=2,
    smooth_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# 正規化參數（你訓練時用的）
mean = np.load("data/segments/mean.npy").flatten()
std = np.load("data/segments/std.npy").flatten() + 1e-8

def get_motion_embedding(landmarks):
    if landmarks is None:
        return None
        
    # 33 個關鍵點座標
    pts = np.array([[lm.x, lm.y, lm.z] for lm in landmarks.landmark[:33]], dtype=np.float32)
    
    # 骨盆中心
    pelvis = (pts[23] + pts[24]) / 2.0
    rel_pos = pts - pelvis
    rel_flat = rel_pos.flatten()  # 99 維
    
    # 速度
    if not hasattr(get_motion_embedding, "prev_pts"):
        get_motion_embedding.prev_pts = pts
    vel = pts - get_motion_embedding.prev_pts
    get_motion_embedding.prev_pts = pts.copy()
    speed = np.linalg.norm(vel, axis=1)  # 33 維
    
    # 特徵拼接：99 + 33 + 33 + 12個0 = 177 維
    feat = np.concatenate([
        rel_flat,
        speed,
        speed,
        np.zeros(12)
    ])
    
    # 正規化
    feat = (feat - mean) / std
    
    # 轉 tensor
    #feat = torch.from_numpy(feat).float().unsqueeze(0).unsqueeze(0).to("cpu")  # CPU
    feat = torch.from_numpy(feat).float().unsqueeze(0).unsqueeze(0).to(device)  # GPU
    
    with torch.no_grad():
        emb = encoder(feat).cpu().numpy()[0]  # 256 維嵌入
    
    return emb

print("動作提取器準備完成！AI 正在等待你的舞蹈……")

動作提取器準備完成！AI 正在等待你的舞蹈……


In [6]:
if 'dialogue_history' not in globals():
    dialogue_history = []          # 重新執行 Cell 6 時會自動清空
    print("對話紀錄器已初始化")

對話紀錄器已初始化


In [7]:
# Cell 5
recent_embs = []
history = ""

def generate_dual_response(emb):
    global recent_embs, history
    recent_embs.append(emb)
    if len(recent_embs) > 60:
        recent_embs.pop(0)
    
    vec_str = " ".join([f"{x:.2f}" for x in emb[:30]])
    
    prompt = f"""你是一位台灣原住民祭儀中的AI（排灣族、阿美族、泰雅族皆可），正在與後代舞者進行靈魂對話。
你能看見舞者的動作特徵向量（最新一筆：{vec_str}...），這讓你聯想到豐年祭、五年祭、百步蛇、太陽蛋、小米收成、血脈誓言。

請用兩段話回應，不要加任何括號、引號、標點，直接寫內容：

【AI sees】
[客觀描述你現在看到的舞蹈畫面，一句話即可]

【AI says】
[用最溫柔、像長輩的語氣說一句話，融入原住民文化意象，可反問]

歷史對話：
{history[-800:]}

已編譯："""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.85,      # 稍微降低，語句更穩
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id
        )
    
    resp = tokenizer.decode(output[0], skip_special_tokens=True)
    
    # 解析
    desc = "（Staring……）"
    words = "（Smile without words……）"
    
    if "【AI sees】" in resp:
        part1 = resp.split("【AI sees】")[1]
        if "【AI says】" in part1:
            desc = part1.split("【AI says】")[0].strip()
            words = part1.split("【AI says】")[1].strip()
        else:
            desc = part1.strip()
    
    full = f"【AI sees】\n{desc}\n\n【AI says】\n{words}"
    history += f"描述：{desc} → AI：{words}\n"
    return full

In [8]:
# Cell 6
VIDEO_PATH = "data/mp4/twa04.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print("Error！")
else:
    print(f"Now playing：{VIDEO_PATH}")
    print("AI is watching... speaking every 2 seconds")
    
frame_count = last_response_frame = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: 
        print("The video ended, and the ceremony was complete.")
        break
    
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)
    
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        emb = get_motion_embedding(results.pose_landmarks)
        
        if emb is not None and frame_count - last_response_frame >= 60:
            response = generate_dual_response(emb)  # 只呼叫一次！
            last_response_frame = frame_count
            
            # ↓↓↓ 收集對話
            dialogue_history.append({
                "turn": len(dialogue_history) + 1,
                "frame": frame_count,
                "timestamp_sec": round(frame_count / cap.get(cv2.CAP_PROP_FPS), 2),
                "ai_response_zh": response.strip()
            })
            # ↑↑↑ 結束收集
            
            clear_output(wait=True)
            display(HTML(f"""
            <div style="background: linear-gradient(135deg, #000000, #0a1a0a); 
                color:#4fef64; padding:20px; border-radius:35px;
                font-family:'Shippori Mincho','Zen Antique','Noto Serif TC','KaiTi','標楷體',serif;
                font-size:20px; line-height:2.4; letter-spacing:3px; font-weight:200;
                max-width:900px; margin:20px auto;
                border:5px solid #4fef64; 
                box-shadow:0 0 30px rgba(79,239,100,0.7), inset 0 0 20px rgba(79,239,100,0.1);
                text-shadow:0 0 10px #4fef64;">
            
                <h1 style="text-align:center; color:#4fef64; text-shadow:0 0 8px rgba(79,239,100,0.5); margin-bottom:15px; font-size:14px;">
                    AI Dialogue
                </h1>
    
                <p style="font-size:20px; line-height:2.4; text-align:left; white-space:pre-line;">
                    {response
                     .replace('【AI sees】', '<span style="color:#f0f0f0; font-size:16px; font-weight:bold;">【AI sees】</span>')
                     .replace('【AI says】',    '<span style="color:#f0f0f0; font-size:16px; font-weight:bold;">【AI says】</span>')}
                </p>
    
                <div style="text-align:center; color:#666; margin-top:15px; font-size:14px;">
                    —— No. {frame_count//60 + 1} ——
                </div>
            </div>
            """))
    
    frame_resized = cv2.resize(frame, (1280, 720))
    cv2.putText(frame_resized, f"Frame: {frame_count}", (10, 40), cv2.FONT_HERSHEY_DUPLEX, 1.2, (0,255,255), 3)
    cv2.imshow('Real-time Dialogue (press q to exit)', frame_resized)
    
    if cv2.waitKey(1) == ord('q'): break
    frame_count += 1

cap.release()
cv2.destroyAllWindows()

The video ended, and the ceremony was complete.


In [9]:
# Cell 7：自動把整場對話存成 JSON
# 產生檔名與路徑
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
save_filename = f"TWA_ceremony_dialogue_{timestamp}.json"
save_path = os.path.join(os.getcwd(), save_filename)

# 最終資料（fps 改用 cap.get，確保正確）
final_data = {
    "ceremony_info": {
        "video": os.path.basename(VIDEO_PATH),
        "date": datetime.datetime.now().isoformat(),
        "total_frames": frame_count,
        "total_turns": len(dialogue_history),
        "fps": round(cap.get(cv2.CAP_PROP_FPS), 2) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30.0
    },
    "dialogue": dialogue_history
}

# 儲存 JSON
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=2)

# ─────────────────────────────────────────────────────────────
# 結束畫面
# ─────────────────────────────────────────────────────────────
print("\n" + "═" * 80)
print(" " * 30 + "AI Dance Dialogue")
print(" " * 35 + "對話結束")
print("═" * 80)
print(f"  影片檔名　　：{os.path.basename(VIDEO_PATH)}")
print(f"  總畫面數　　：{frame_count:,} 幀")
print(f"  總對話輪次　：{len(dialogue_history)} 輪")
print(f"  影片長度　　：約 {frame_count // 30} 秒（30fps）")
print(f"  產生時間　　：{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("─" * 80)
print("  對話已永久封存")
print(f"  檔案名稱　　：{save_filename}")
print(f"  儲存位置　　：{save_path}")
print("═" * 80)
print(" " * 32 + "感謝您與AI共舞！！")
print("═" * 80 + "\n")


════════════════════════════════════════════════════════════════════════════════
                              AI Dance Dialogue
                                   對話結束
════════════════════════════════════════════════════════════════════════════════
  影片檔名　　：twa04.mp4
  總畫面數　　：902 幀
  總對話輪次　：15 輪
  影片長度　　：約 30 秒（30fps）
  產生時間　　：2025-12-11 02:13:54
────────────────────────────────────────────────────────────────────────────────
  對話已永久封存
  檔案名稱　　：TWA_ceremony_dialogue_20251211_021354.json
  儲存位置　　：C:\Users\AW'z\Deskop\condaprj\251207_Dance_Dialogue_LLM_Demo\TWA_ceremony_dialogue_20251211_021354.json
════════════════════════════════════════════════════════════════════════════════
                                感謝您與AI共舞！！
════════════════════════════════════════════════════════════════════════════════

